# 02 - The recommender + a like/dislike classifier

Two parts:
1. **Recommender** - average a few 'liked' home vectors into a taste vector, then rank all homes by cosine similarity.
2. **Classifier** - a small logistic regression that learns like/dislike from the style vectors, to get one honest accuracy number.

Run `python prepare_data.py` first.

In [ ]:
import sys; sys.path.append('..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import os

from recommender import build_taste_profile, recommend

embeddings = np.load('../artifacts/embeddings.npy')
listings = pd.read_csv('../artifacts/listings.csv')
embeddings.shape, listings.shape

## 1. Recommender demo
Pretend the buyer liked these 3 homes:

In [ ]:
liked = [0, 1, 2]
fig, axes = plt.subplots(1, 3, figsize=(11, 3))
for ax, i in zip(axes, liked):
    ax.imshow(Image.open(os.path.join('../artifacts/images_sample', listings.loc[i, 'image'])))
    ax.axis('off'); ax.set_title('liked')
plt.show()

In [ ]:
taste = build_taste_profile(embeddings, liked)
recs = recommend(embeddings, listings, taste, exclude_idx=liked, top_n=6)
recs[['id', 'price', 'bedrooms', 'builder', 'match_score']]

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(16, 3))
for ax, (_, row) in zip(axes, recs.iterrows()):
    ax.imshow(Image.open(os.path.join('../artifacts/images_sample', row['image'])))
    ax.axis('off'); ax.set_title(f"{row['match_score']:.2f}")
plt.show()

## 2. Like/dislike classifier (one honest number)

We have no real click logs in this public sample, so we test the idea with a **clearly-defined stand-in preference**: *'this buyer prefers higher-priced homes'* (like = price above the median). The classifier only sees the **image style vectors**, never the price - so the accuracy tells us how well visual style alone predicts that preference.

This is an illustrative evaluation, stated as such; the live app trains the same classifier on the user's real clicks instead.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

y = (listings['price'] > listings['price'].median()).astype(int)
X_tr, X_te, y_tr, y_te = train_test_split(embeddings, y, test_size=0.2, random_state=42, stratify=y)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_tr, y_tr)
acc = accuracy_score(y_te, clf.predict(X_te))
print(f'Held-out accuracy: {acc:.1%}')

Style vectors alone predict the stand-in preference well above the 50% coin-flip baseline - evidence the image features carry real preference signal, which is exactly what the recommender relies on.